## Preprocesamiento y Modelado

Una vez inspeccionado el dataset en `customer_churn_eda.ipynb` definimos una una estrategia de preprocesamiento iterativo (de menos a más) para el encontrar

In [126]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import OneHotEncoder,RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, make_scorer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.discriminant_analysis import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from feature_engine.imputation import RandomSampleImputer


### Configuración de constantes, rutas y variables 

En esta sección definimos constantes, rutas de archivos y atributos del dataset

In [127]:
# Rutas de los archivos de datos
TRAIN_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/train.csv"
TEST_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Cargamos los datos
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Definir variables objetivo
TARGET = 'Exited'
# Variables numéricas: Incluyo las continuas y las binarias numéricas (HasCrCard, IsActiveMember)
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
# Variables categóricas: Las de texto con pocas categorías
CAT_FEATURES = ['Geography', 'Gender', 'Surname']
# Variables a eliminar inicialmente (IDs y apellido)
DROP_FEATURES = ['CustomerId']
SURNAME_COL = 'Surname'
RANDOM_STATE = 100            # Semilla para reproducibilidad

### 1. Preparación de Datos

Separamos variables independientes y dependientes en X_train e y_train por convención.

In [128]:
# X_train = variables independientes
# y_train = variable dependiente u objetivo
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

---------------------------------------

### Funciones auxiliar
#### Construir preprocesadores

In [129]:
def make_preprocessor(X: pd.DataFrame, version: str):
    """Genera un ColumnTransformer con pipelines de preprocesamiento
    para variables numéricas y categóricas según la versión indicada.

    Args:
        X (pd.DataFrame): _input data frame_
        version (str): Versión del preprocesamiento en formato 'N#_C#'
        e.g. 'N3_C1' donde N# indica la versión numérica y C# la categórica
        numerical_cols (_type_): columnas numéricas para el preprocesamiento
        categorical_cols (_type_): columnas categóricas para el preprocesamiento

    Raises:
        ValueError: _unknown numeric version_
        ValueError: _unknown categorical version_

    Returns:
        _type_: ColumnTransformer con pipelines de preprocesamiento
    """
    num_version, categorical_version = version.split("_")  # e.g. 'N3', 'C1'

    # --- Numerical pipeline ---
    numerical_transformers = []

    if num_version == "N1":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con mediana
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N2":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con mediana + indicador de faltantes
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N3":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana + indicador de faltantes + escalado estandar
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N4":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con mediana + escalado estandar
        # Escalado robusto a outliers con RobustScaler
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N5":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        num_pipe = Pipeline(steps=[
            ('scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform"))
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N6":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        # Escalado robusto a outliers con RobustScaler
        num_pipe = Pipeline(steps=[
            ('pre_scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform")),
            ("scaler", RobustScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N7":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado RobustScaler
        # KNN requiere escalar primero para calcular distancias bien.
        num_pipe = Pipeline(steps=[
            ('scaler', RobustScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform")),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    
    elif num_version == "N8":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        num_pipe = Pipeline(steps=[
            ('scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=9, weights="uniform"))
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N9":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        # añadir indicador puede ayudar si hubiera correlación entre faltantes y target
        num_pipe = Pipeline(steps=[
            ('pre_scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="uniform",add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N10":
        # Imputamos todas las variables numéricas igual
        numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
        # Imputa valores faltantes con KNN + escalado estandar
        # KNN requiere escalar primero para calcular distancias bien.
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        # vecinos más cercanos pesan más puede que la imputación sea más fina (si hay grupos claros)
        num_pipe = Pipeline(steps=[
            ('pre_scaler', StandardScaler()), 
            ("imputer", KNNImputer(n_neighbors=5, weights="distance",add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N11":
        # Imputamos cada variable numérica por separado según su distribución
        # CreditScore= 20.51% missing -> Distribución normal -> SimpleImputer mediana
        # Age= 0.00% missing -> Distribución normal -> SimpleImputer mediana
        # Tenure= 0.00% missing -> Distribución plana -> RandomSampleImputer (de las observaciones existentes)
        # Balance= 20.19% missing -> Distribución binomial -> SimpleImputer constante 0 (asumimos que falta significa balance 0,abre cuenta sin saldo)
        # NumOfProducts= 12.20% missing -> SimpleImputer constante 0 (asumimos que falta significa no tener productos,abre cuenta sin productos)
        # EstimatedSalary= 10.40% missing -> Distribución plana -> RandomSampleImputer (de las observaciones existentes) 
        # Surname= 5.11% missing -> Categorical -> SimpleImputer unknown + OneHotEncoder
        # HasCrCard= 4.69% missing -> SimpleImputer constante 0 (asumimos que falta significa no tener tarjeta,abre cuenta sin solicitar tarjeta)
        # IsActiveMember= 0% missing -> Binaria -> SimpleImputer constante 0 (asumimos que falta significa no ser miembro activo)
        num_pipe_median = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")), # imputación mediana
            ("scaler", RobustScaler()), # escalado robusto a outliers
        ])
        
        num_pipe_random_sample = Pipeline(steps=[
            # random_state : int, variable name or a list of variables to determine the seed, observation per observation.
            #seed : 'general' (one seed will be used to impute the entire dataframe) or 'observation' (the seed will be set for each observation using the values of the variables indicated)
            ("imputer", RandomSampleImputer(random_state=RANDOM_STATE,seed='general')), # imputación por muestra aleatoria
            ("scaler", RobustScaler()), # escalado robusto a outliers
        ])
        num_pipe_constant_0 = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ])
        numerical_transformers.append(("median", num_pipe_median, ["CreditScore","Age"]))
        numerical_transformers.append(("random_sample", num_pipe_random_sample, ["Tenure","EstimatedSalary"]))
        numerical_transformers.append(("constant_0", num_pipe_constant_0, ["Balance","NumOfProducts","HasCrCard","IsActiveMember"]))
    else:
        raise ValueError(f"Unknown numeric version: {num_version}")

    # --- Categorical pipeline (si aplica) ---
    categorical_transformers = []
    if categorical_version == "C0": 
        # No aplica pipeline en categoricas
        pass
    elif categorical_version == "C1":
        # Categóricas normales: imputar + onehot
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        categorical_transformers.append(("cat", categorical_pipe, ['Geography', 'Gender']))
    elif categorical_version == "C11":
        # Continuación de num_version == "N11"
        # Surname= 5.11% missing -> Categorical -> SimpleImputer unknown + OneHotEncoder
        print("Categorical version C11: special handling for Surname column")
        cat_pipe_surname = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ])
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        categorical_transformers.append(("surname", cat_pipe_surname, ["Surname"]))
        categorical_transformers.append(("cat", categorical_pipe, ["Geography", "Gender"]))   

    else:
        raise ValueError(f"Unknown categorical version: {categorical_version}")

    # --- Build ColumnTransformer ---
    # Transformers numéricos y categóricos
    transformers = []
    transformers.extend(numerical_transformers)     
    transformers.extend(categorical_transformers)

    # Construimos el ColumnTransformer final
    # que une pipelines numéricos + pipelines categóricos
    col_trans_preprocessor = ColumnTransformer(
        transformers=transformers, # lista de tuplas (name, pipeline, cols)
        remainder="drop", # elimina columnas no especificadas
        verbose_feature_names_out=True) # nombres detallados de columnas
    return col_trans_preprocessor


#### Construir pipelines

In [ ]:
# Creación y evaluación del pipeline
# Construcción del pipeline con preprocesador y modelo

def make_pipeline(preprocessor: ColumnTransformer, model=None):
    """Construye un Pipeline con el preprocesador y el modelo indicado.
    Args:
        preprocessor (ColumnTransformer): Preprocesador ColumnTransformer
        model (_type_, optional): Modelo de clasificación. Defaults to None.
    Returns:
        Pipeline: Pipeline con preprocesador y modelo, si no se indica modelo
        se usa LinearDiscriminantAnalysis por defecto.
    """
    if model is None:
        model = LinearDiscriminantAnalysis()
        #model = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])




#### Evaluar pipelines

Esta función evalua el pipeline usando validación cruzada estratificada

In [131]:
def evaluate_pipeline(pipe: Pipeline, X: pd.DataFrame, y: pd.Series, n_splits=5):
    """ Evalúa el pipeline usando Validación Cruzada estratificada
    Args:
        pipe (Pipeline): Pipeline a evaluar.
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        dict: Diccionario con las métricas promedio y desviación estándar.
    """
    # Configuramos la Validación Local Cruzada  n_splits splits (divisiones)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    # Definimos las métricas que queremos extraer
    # f1, roc_auc, precision, recall, accuracy son strings estándar de sklearn.
    # Kappa requiere make_scorer.
    scoring_metrics = {
        "f1": "f1",
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        'kappa': make_scorer(cohen_kappa_score),
        'precision': 'precision',
        'recall': 'recall',
    }
    cv_results = cross_validate(pipe, X, y, cv=cv, scoring=scoring_metrics, n_jobs=-1)
    return {
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std":  cv_results["test_f1"].std(),
        "auc_mean": cv_results["test_roc_auc"].mean(),
        "auc_std":  cv_results["test_roc_auc"].std(),
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std":  cv_results["test_accuracy"].std(),
        "kappa_mean": cv_results["test_kappa"].mean(),
        "kappa_std":  cv_results["test_kappa"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std":  cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std":  cv_results["test_recall"].std()
    }

#### Evaluar varios modelos

In [132]:

def benchmark_models_with_fixed_preprocess(X: pd.DataFrame, y: pd.Series, models: dict, 
                                           best_preprocesor_version: str, n_splits=5):
    """Evalúa varios modelos con un preprocesador fijo usando Validación Cruzada.
    Args:
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        models (dict): Diccionario con nombre y objeto del modelo a evaluar.
        best_preprocesor_version (str): Versión del preprocesador a usar.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        pd.DataFrame: DataFrame con resultados de cada modelo evaluado.
    """
    # Construimos el preprocesador fijo con la mejor versión
    best_preprocesor = make_preprocessor(X, best_preprocesor_version)

    model_results = []  # Lista para almacenar resultados de cada modelo
    # Evaluamos cada modelo con el preprocesador fijo
    for name, model in models.items():
        # Construimos el pipeline con preprocesador fijo y el modelo actual
        pipe = Pipeline([("preprocessor", best_preprocesor), ("classifier", model)])
        try:
            # Evaluamos el pipeline con validación cruzada para el pipeline actual
            cv_metrics = evaluate_pipeline(pipe, X, y , n_splits=n_splits)
            model_results.append({
                "model": name,
                "preprocessor_version": best_preprocesor_version,
                **cv_metrics
            })
        except Exception as e:
            model_results.append({"model": name, "error": str(e)})
    # Construimos el DataFrame de resultados ordenado por F1 medio
    out = pd.DataFrame(model_results).sort_values(by="f1_mean", ascending=False, na_position="last")
    return out


------------------
## Evaluación de diferentes preprocesadores y modelos

A partir de aquí comenzamos la evaluación de los distintos preprocesadores que se han configurado en la función `make_preprocesor` y modelos, configurador `benchmark_models_with_fixed_preprocess`

In [133]:
# Experimentos: combinaciones de preprocesamiento a probar

from xgboost import XGBClassifier


EXPERIMENTS = [
    #"N1_C0",  # num only (simple imputer mediana)
    #"N2_C0",  # num only (simple imputer mediana) + indicator
    #"N3_C0",  # num only (simple imputer mediana) + indicator + scaler
    "N3_C1",  # num only (simple imputer mediana + indicator + scaler) + cat (onehot) sin surname
    "N4_C1",  # num (robust scaler) + cat (onehot) sin surname
    "N5_C1",  # num (KNN imputer 5 vecinos) + cat (onehot) sin surname
    "N6_C1",  # num (KNN imputer + robust scaler) + cat (onehot) sin surname
    #"N7_C1",  # num (KNN imputer + robust scaler) + cat (onehot) sin surname
    #"N8_C1",  # num (KNN imputer 9 vecinos) + cat (onehot) sin surname
    # No mejora -> Volvemos a la base N5_C1
    "N9_C1",  # num (KNN imputer 5 vecinos + indicador) + cat (onehot) sin surname
    "N10_C1",  # num (KNN imputer 5 vecinos + indicador + pesos distancia) + cat (onehot) sin surname
    "N11_C1",  # num (imputación por variable según distribución + RandomSampleImputer) + cat (surname imputación unknown + onehot)
]

# Modelos a probar
models = {
    # Modelos lineales
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"),
    # Arboles
    # No necesitan escalado de variables
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced",n_jobs=-1,),
    "RandomForest_bal_subsample": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced_subsample",n_jobs=-1,),
    # Otros modelos
    "NaiveBayes": GaussianNB(),
    "RedesNeurales": MLPClassifier(hidden_layer_sizes=(50,30), max_iter=1000, random_state=RANDOM_STATE,
                                   activation='relu',solver='adam',early_stopping=True),
    "KNN_5": KNeighborsClassifier(n_neighbors=5, n_jobs=-1,),
    "KNN_5_distance": KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),

    
}


#### Búsqueda del mejor pipeline

Evaluamos las diferentes configuraciones de preprocesador que tenemos configuradas con el modelo por defecto definido en la función `make_pipeline` con el objetivo de obtener mejor versión o combinación de preprocesado. 

In [ ]:
preprocesors_results = []
for experiment_version in EXPERIMENTS:
    best_preprocesor = make_preprocessor(train_df, experiment_version)
    
    pipe = make_pipeline(best_preprocesor) 
    print("Evaluando preprocesador versión:", experiment_version)
    cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
    preprocesors_results.append({"version": experiment_version, **cv_metrics})
    #print(experiment_version, cv_metrics)

results_df = pd.DataFrame(preprocesors_results).sort_values("f1_mean", ascending=False)
display(results_df)
best_preprocesor_version = results_df.iloc[0]["version"]
print("Mejor versión de preprocesador:", best_preprocesor_version)

Evaluando preprocesador versión: N3_C1 LinearDiscriminantAnalysis
Evaluando preprocesador versión: N3_C1 LogisticRegression
Evaluando preprocesador versión: N3_C1 DecisionTree
Evaluando preprocesador versión: N3_C1 RandomForest
Evaluando preprocesador versión: N3_C1 RandomForest_bal_subsample
Evaluando preprocesador versión: N3_C1 NaiveBayes
Evaluando preprocesador versión: N3_C1 RedesNeurales
Evaluando preprocesador versión: N3_C1 KNN_5
Evaluando preprocesador versión: N3_C1 KNN_5_distance
Evaluando preprocesador versión: N4_C1 LinearDiscriminantAnalysis
Evaluando preprocesador versión: N4_C1 LogisticRegression
Evaluando preprocesador versión: N4_C1 DecisionTree
Evaluando preprocesador versión: N4_C1 RandomForest
Evaluando preprocesador versión: N4_C1 RandomForest_bal_subsample
Evaluando preprocesador versión: N4_C1 NaiveBayes
Evaluando preprocesador versión: N4_C1 RedesNeurales
Evaluando preprocesador versión: N4_C1 KNN_5
Evaluando preprocesador versión: N4_C1 KNN_5_distance
Evaluand

/home/administrador/miniforge3/envs/tareaK_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/administrador/miniforge3/envs/tareaK_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/administrador/miniforge3/envs/tareaK_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier,

Evaluando preprocesador versión: N11_C1 RedesNeurales
Evaluando preprocesador versión: N11_C1 KNN_5
Evaluando preprocesador versión: N11_C1 KNN_5_distance


,version,model,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
42,N9_C1,RedesNeurales,0.538358,0.016189,0.842400,0.006271,0.849750,0.002519,0.455441,0.015383,0.720247,0.017024,0.430675,0.024509
51,N10_C1,RedesNeurales,0.532540,0.013335,0.842730,0.005980,0.850000,0.003010,0.451074,0.009262,0.733365,0.040003,0.420245,0.028047
13,N4_C1,RandomForest_bal_subsample,0.526889,0.013753,0.833024,0.009845,0.849000,0.003177,0.445140,0.014599,0.728359,0.011691,0.412883,0.014851
58,N11_C1,RandomForest_bal_subsample,0.526884,0.010214,0.838049,0.005256,0.850125,0.001696,0.446417,0.009697,0.738510,0.011169,0.409816,0.013800
3,N3_C1,RandomForest,0.525520,0.024641,0.833526,0.009595,0.849875,0.005935,0.445037,0.026478,0.737427,0.022936,0.408589,0.025101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54,N11_C1,LinearDiscriminantAnalysis,0.319523,0.019999,0.767890,0.006647,0.806500,0.005025,0.230784,0.019328,0.565510,0.035480,0.223313,0.018548
62,N11_C1,KNN_5_distance,0.288210,0.015966,0.652352,0.011464,0.760000,0.007448,0.151518,0.017955,0.364950,0.020530,0.238650,0.016621
61,N11_C1,KNN_5,0.278010,0.020223,0.660059,0.007204,0.786750,0.003674,0.173626,0.019162,0.447894,0.020075,0.201840,0.018241
60,N11_C1,RedesNeurales,0.203797,0.054966,0.515010,0.012275,0.811250,0.003771,0.156101,0.041235,0.748469,0.089410,0.120859,0.039503


Mejor versión de preprocesador: N9_C1


#### Búsqueda del mejor modelo

Evaluamos los modelos con el mejor preprocesador encontrado 

In [135]:
# Evaluamos los modelos con el mejor preprocesador encontrado
models_df = benchmark_models_with_fixed_preprocess(X_train, y_train, models, best_preprocesor_version, n_splits=5)
print("---- Resultados de validación cruzada de modelos con preprocesador fijo:----")
display(models_df)

best_model_name = models_df.iloc[0]["model"]
print("Mejor versión de preprocesador:", best_preprocesor_version)
print("Mejor modelo:", best_model_name)

---- Resultados de validación cruzada de modelos con preprocesador fijo:----


,model,preprocessor_version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
6,RedesNeurales,N9_C1,0.538358,0.016189,0.842400,0.006271,0.849750,0.002519,0.455441,0.015383,0.720247,0.017024,0.430675,0.024509
4,RandomForest_bal_subsample,N9_C1,0.521740,0.012624,0.837119,0.006483,0.850500,0.001146,0.442707,0.011320,0.749430,0.010894,0.400613,0.017309
3,RandomForest,N9_C1,0.519924,0.011068,0.838054,0.006047,0.850750,0.001212,0.441523,0.010127,0.754437,0.008509,0.396933,0.014596
1,LogisticRegression,N9_C1,0.496977,0.013376,0.770888,0.009400,0.714750,0.005009,0.319267,0.016549,0.387783,0.008212,0.692025,0.027682
2,DecisionTree,N9_C1,0.468057,0.021319,0.666064,0.013241,0.782875,0.010090,0.331699,0.027406,0.467693,0.024027,0.468712,0.021464
8,KNN_5_distance,N9_C1,0.467116,0.016809,0.771117,0.003382,0.827500,0.005214,0.372351,0.019243,0.630415,0.022288,0.371166,0.015520
7,KNN_5,N9_C1,0.464598,0.014972,0.770265,0.003973,0.828625,0.004766,0.371544,0.017015,0.639548,0.022467,0.365031,0.014388
5,NaiveBayes,N9_C1,0.437761,0.016447,0.771120,0.008575,0.805000,0.007624,0.324175,0.021406,0.531340,0.027028,0.372393,0.012053
0,LinearDiscriminantAnalysis,N9_C1,0.331837,0.015740,0.770065,0.009218,0.808375,0.004268,0.242534,0.015610,0.574326,0.029009,0.233742,0.014570


Mejor versión de preprocesador: N9_C1
Mejor modelo: RedesNeurales


## Construcción del pipeline final para Kaggle

Una vez obtenido la mejor combinación de preprocesadores y el mejor modelo, construimos el pipeline final para kaggle con la mejor combinación de ambos y volvemos a ejecutar la evaluación y el entrenamiento para finalmente obtener la predicción y generar el fichero para kaggle. 

In [137]:
# Construcción del pipeline final para Kaggle con el mejor preprocesador y modelo
# Construimos el preprocesador fijo con la mejor versión
best_preprocesor = make_preprocessor(X_train, best_preprocesor_version)
best_model = models[best_model_name]
# Pipeline Completo (Preprocesamiento + Modelo)
best_model_pipeline = Pipeline(steps=[
    ('preprocessor', best_preprocesor),
    ('classifier', best_model)
])
# Configuramos y ejecutamos la Validación Cruzada local
cv_metrics = evaluate_pipeline(best_model_pipeline, X_train, y_train, n_splits=5)
# Generación de Submission para Kaggle con el mejor modelo encontrado
# Re-entrenamos con TODOS los datos de train para la predicción final
best_model_pipeline.fit(X_train, y_train) 
test_predictions = best_model_pipeline.predict(test_df)

# Crear fichero de salida
submission_df = pd.DataFrame({
    'CustomerId': test_df['CustomerId'],
    'Exited': test_predictions
})
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Fichero '{SUBMISSION_PATH}' generado correctamente.")

print("\n---- Mejores Resultados y Validación Cruzada local -----")
print("Mejor modelo:", best_model_name)
print("Mejor versión de preprocesador:", best_preprocesor_version)
print(f"Mean F1-Score:  {cv_metrics['f1_mean']:.4f} (+/- Std {cv_metrics['f1_std']:.4f})")
print(f"Mean Accuracy:  {cv_metrics['accuracy_mean']:.4f} (+/- Std {cv_metrics['accuracy_std']:.4f})")
print(f"Mean Kappa:     {cv_metrics['kappa_mean']:.4f}")
print(f"Mean Precision: {cv_metrics['precision_mean']:.4f}")
print(f"Mean Recall:    {cv_metrics['recall_mean']:.4f}")



Fichero '/kaggle/working/submission.csv' generado correctamente.

---- Mejores Resultados y Validación Cruzada local -----
Mejor modelo: RedesNeurales
Mejor versión de preprocesador: N9_C1
Mean F1-Score:  0.5384 (+/- Std 0.0162)
Mean Accuracy:  0.8498 (+/- Std 0.0025)
Mean Kappa:     0.4554
Mean Precision: 0.7202
Mean Recall:    0.4307
